# Typing in Param

Param provides a rich runtime model for validating values, and starting in version 2.4.0 it also works well with Python type checkers.

This guide covers:

- how types are inferred from Parameter *types* and constructor *arguments*
- where typing support is heading next (annotation-first parameter declarations)

Before we dive into the details it's important to clarify the difference between the type annotation and parameter declaration:

- **type annotation** communicates intent to static tools (`Literal`, unions, etc.)
- **Parameter declaration** controls runtime behavior and validation

Throughout this guide, examples use standard `param.Parameterized` classes, and type hints are meant for tools like Pyright and mypy.

In [ ]:
import param
import typing as t
from typing import Any, Literal
from typing_extensions import assert_type

## Type inference from Parameter types and arguments

Param parameter classes encode value constraints (runtime). Thanks to Python's type system and the ability to define overloads that perform type narrowing it is often possible to infer the underlying type from the Parameter declaration.

In general:

- the Parameter subclass provides the base type (e.g. `Integer` -> `int`, `String` -> `str`)
- keyword arguments can refine the type (e.g. `allow_None=True`, `item_type=int`, `class_=MyType`)
- type checkers then infer the type of instance attributes accordingly

In [ ]:
class InferredTypes(param.Parameterized):
    title = param.String()
    retries = param.Integer(allow_None=False)
    timeout = param.Number(allow_None=True)

    # item_type refines list element types
    tags = param.List(item_type=str)

class Model:
    pass

class MoreInferredTypes(param.Parameterized):
    model = param.ClassSelector(class_=Model, allow_None=False, default=Model())
    optional_model = param.ClassSelector(class_=Model, allow_None=True, default=None)


i = InferredTypes()
m = MoreInferredTypes()

# These are checked by static type checkers (runtime no-op assertions):
assert_type(i.title, str)
assert_type(i.retries, int)
assert_type(i.timeout, int | float | None)
assert_type(i.tags, list[str])

assert_type(m.model, Model)
assert_type(m.optional_model, Model | None)

"Type assertions executed (static checking is done by your type checker)."

In the example above:

- `param.String()` gives `str`
- `param.Integer()` gives `int`
- `param.Number(allow_None=True)` gives `int | float | None`
- `param.List(item_type=str)` gives `list[str]`
- `param.ClassSelector(class_=Model, ...)` gives `Model` (or `Model | None` when `allow_None=True`)

This means Param metadata and constructor arguments can directly improve the quality of type inference.

## Exceptions & Limitations

### Type Narrowing

Unfortunately type narrowing can only get you so far in Python's type system. Specifically `Literal` values, as in when a `Selector` parameter is bound on a `Parameterized` class cannot be automatically inferred from the `objects`, e.g.:

In [ ]:
class SelectorFromLiterals(param.Parameterized):
    mode = param.Selector(objects=["train", "eval"])

assert_type(m.model, Any);

For the time being we need to add a redundant type annotation (and add a corresponding `type: ignore`):

In [ ]:
class SelectorFromLiterals(param.Parameterized):
    mode: Literal["train", "eval"] = param.Selector(
        objects=["train", "eval"]
    )  # type: ignore[assignment]

assert_type(m.model, Literal["train", "eval"]);

This is unfortunate and one reason why this is not the end to the typing story for Param.

### Signature

The second problem with the current approach is that it does not automatically infer that parameters are arguments to the constructor. Currently the signature is therefore automatically added at runtime, e.g. to allow tab-completion in contexts such as notebooks and IPython.

To demonstrate this, here's what the runtime signature looks like:

In [ ]:
import inspect

inspect.signature(InferredTypes)

However a type checker will not be able to infer that `Parameters` are arguments to the constructor automatically.

Both of these issues will be addressed by future proposals.

## Future direction

As discussed above, the current approach to typing in Param has limits. While the proposal is still being iterated on, the prototype [PR #1066](https://github.com/holoviz/param/pull/1066) introduces a new base class that will allow inferring parameter types from the type annotations.

While the precise naming is still being decided the future will look something like this:

```python
class MyModel(param.ParamModel):
    # annotation-only -> synthesized automatically
    name: str
    count: int = 0
    ratio: float | None = None
    tags: list[str]
    mode: Literal["fast", "slow"] = "fast"

    # explicit parameter config via param.Param factory
    label: str = param.ParamField(default="untitled", doc="Human-readable label")

    # explicit parameter type override
    path: str = param.ParamField(parameter=param.String, regex=r"^/")

    # regular class attribute
    attr: ClassVar[str] = "foo"
```

This approach will align Param more closely with modern tooling such as dataclasses and pydantic, reduce boiler plate, and finally provide a fully typed signature.

## Practical recommendations

For now these are our practical recommendations for typing in Param:

- Use specific Parameter classes (`Integer`, `String`, `ClassSelector`, etc.) instead of generic `Parameter` whenever possible.
- Use Parameter keyword arguments (`allow_None`, `item_type`, `class_`, `bounds`) to improve both runtime checks and type precision.
- For finite choices today, prefer `Literal[...]` annotations with `Selector`.
- Run your type checker (Pyright/mypy) alongside your tests to get both static and runtime safety.